# ECG Sleep-Apnea Detection with AlexNet Transfer Learning (Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harishbala123/ecg-transfer-learning-classification/blob/main/notebooks/colab_training.ipynb)

PhysioNet Apnea-ECG: single-lead ECG → RR-interval / ECG-derived-respiration wavelet images → ImageNet-pretrained AlexNet (weights ported from torchvision). Subject-level split. Research code, not a medical device.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The notebook clones the repository and runs the same scripts used on Windows, so the code is identical in both environments.
Colab already ships TensorFlow/Keras, so only the missing packages are installed. This avoids replacing Colab's CUDA-matched TensorFlow build.

In [ ]:
!nvidia-smi -L || echo 'No GPU: select Runtime > Change runtime type > T4 GPU'

In [ ]:
import os
if not os.path.exists('/content/ecg-transfer-learning-classification'):
    !git clone -q https://github.com/Harishbala123/ecg-transfer-learning-classification.git /content/ecg-transfer-learning-classification
%cd /content/ecg-transfer-learning-classification
!git log -1 --oneline

In [ ]:
!pip install -q pyyaml wfdb PyWavelets opencv-python-headless
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

## 1. Download + verify the Apnea-ECG database (SHA-256)

In [ ]:
!python scripts/download_data.py

## 2. Port ImageNet AlexNet weights from torchvision (Colab has PyTorch pre-installed)

In [ ]:
!python scripts/export_alexnet_weights.py

## 3. ECG → per-minute RR/EDR wavelet images

In [ ]:
!python scripts/build_dataset.py

## 4. Two-stage transfer learning

In [ ]:
!python scripts/train.py

## 5. Evaluate on held-out subjects

In [ ]:
!python scripts/evaluate.py
!cat results/reports/test_classification_report.txt

In [ ]:
!python scripts/predict_record.py --record x05

In [ ]:
from IPython.display import Image, display
display(Image('results/graphs/training_curves.png'))
display(Image('results/graphs/confusion_matrix_minutes.png'))
display(Image('results/graphs/roc_curve.png'))
display(Image('results/graphs/record_apnea_burden.png'))
display(Image('results/graphs/sample_predictions.png'))

In [ ]:
!pip install -q pytest && python -m pytest -q

## Export artifacts
The zip holds the trained model(s), metrics and graphs. Unzip it into the local project folder (Windows) to use
the Colab-trained model with the local evaluation and demo scripts. Mounting Google Drive is optional and only
needed if you want a persistent copy; no credentials are stored in the notebook.

In [ ]:
!zip -qr /content/ecg-transfer-learning-classification_artifacts.zip models/alexnet_apnea.keras results && ls -lh /content/ecg-transfer-learning-classification_artifacts.zip

In [ ]:
SAVE_TO_DRIVE = False   # set True to copy the zip to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp /content/ecg-transfer-learning-classification_artifacts.zip /content/drive/MyDrive/
else:
    from google.colab import files
    files.download('/content/ecg-transfer-learning-classification_artifacts.zip')